### Installations and imports:

In [ ]:
!pip install -U bitsandbytes  # for using quantized weights (requires restart)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 77.4 MB/s  0:00:00


Restart session here, then can continue running below cells:

In [ ]:
# Flash attention installation stopped working under current Colab environment (using SDPA instead):
# !pip install ninja  # for fast compilation during flash-attn install
# !pip install flash-attn --no-build-isolation  # for using Flash Attention

# !pip install transformers
# !pip install torch

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


from typing import Literal
import json
import re
import os
import shutil

In [ ]:
from google.colab import files, drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


### Util:

In [ ]:
# ================================
#        Files I/O methods:
# ================================

def json_load(json_path):
    with open(json_path, mode="r", encoding="utf-8") as f:
        return json.load(f)

def json_list_write(lst, json_path, is_pretty_print_json=True, is_print_mssg=True):
    if len(lst) == 0:
        return False

    if is_print_mssg:
        print(f'Writing to [{json_path}]')

    obj_json_str = json.dumps(lst, indent=4 if is_pretty_print_json else None)

    file_write(obj_json_str, json_path)
    return True


def file_read(json_path):
    with open(json_path, mode="r", encoding="utf-8") as f:
        return f.read()

def file_write(text, filepath):
    with open(filepath, mode="w", encoding="utf-8") as f:
        return f.write(text)


# --- Google Drive: ---

content_base_path = '/content'
drive_base_path = f'{content_base_path}/drive/MyDrive'

def copy_to_drive(from_filepath, to_folder_path='colab_output'):
  shutil.copy(
      f"{content_base_path}/{from_filepath}",
      f"{drive_base_path}/{to_folder_path}/")

def read_from_drive(from_filepath, from_folder_path='colab_input'):
    json_path = f'{drive_base_path}/{from_folder_path}/{from_filepath}'
    return file_read(json_path)

def json_load_from_drive(from_filepath, from_folder_path='colab_input'):
    json_path = f'{drive_base_path}/{from_folder_path}/{from_filepath}'
    return json_load(json_path)



# ================================
#             Misc:
# ================================

from tqdm.auto import tqdm

def get_progress_bar(lst, description= None):
    progress_bar = tqdm(lst)
    if description is not None:
        progress_bar.set_description(description)
    return progress_bar

### Constants and data:

In [ ]:
# MODEL_NAME = "Snowflake/Arctic-Text2SQL-R1-7B"
MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"

datatype: Literal["train", "valid", "test"] = "test"

In [ ]:
# SQLFuse schema encoding/linking data:
sqlfuse_schema_info_filepath = 'sqlfuse_schema_info.json'
sqlfuse_schema_links_filepath = f'sqlfuse_ehrsql2024_{datatype}.json'

sqlfuse_schemas_info = json_load_from_drive(sqlfuse_schema_info_filepath)
sqlfuse_schema_links = json_load_from_drive(sqlfuse_schema_links_filepath)

In [ ]:
# Dataset:
dataset_filepath = f'ehrsql2024_tsql_qpl_cte_{datatype}.json'

In [ ]:
# Temporal guidance and user-methods specification:
temp_guide_sqlite_filepath = 'temporal_methods_specificatios_llm_oriented_v3___sqlite.md'
temp_guide_tsql_no_user_func_filepath = 'temporal_methods_specificatios_llm_oriented_v3___tsql_nu.md'
temp_guide_tsql_with_user_func_filepath = 'temporal_methods_specificatios_llm_oriented_v3___tsql_u.md'

temp_guide_sqlite = read_from_drive(temp_guide_sqlite_filepath)
temp_guide_tsql_no_user_func = read_from_drive(temp_guide_tsql_no_user_func_filepath)
temp_guide_tsql_with_user_func = read_from_drive(temp_guide_tsql_with_user_func_filepath)

### Model:

In [ ]:
def load_model(is_4bit=True):
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        use_fast=True
    )

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        quantization_config=quantization_config if is_4bit else None,
        attn_implementation="sdpa"  # "flash_attention_2"
        # low_cpu_mem_usage=True
    )

    model.eval()
    return tokenizer, model

def model_get_ans(prompt, tokenizer, model, max_new_tokens):
    inputs = tokenizer(
        prompt,
        return_tensors="pt" #,
        # truncation=True
    ).to("cuda")  # .to(model.device) ?

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            top_p=1.0,  #  < 0.5: more deterministic output for technical tasks
            eos_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
    return decoded


def get_amount_of_tokens(txt, tokenizer):
  return len(tokenizer.encode(txt))

In [ ]:
tokenizer, model = load_model(is_4bit=False)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
print(f"Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

Allocated: 15.23 GB
Reserved: 15.25 GB


In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "How to be a better AI researcher?"}
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print(prompt)
print(get_amount_of_tokens(prompt, tokenizer))

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
How to be a better AI researcher?<|im_end|>
<|im_start|>assistant

27


In [ ]:
print(model_get_ans(prompt, tokenizer, model, 1024))

system
You are a helpful assistant.
user
How to be a better AI researcher?
assistant
Becoming a better AI researcher requires dedication, hard work, and continuous learning. Here are some tips that can help you improve your skills as an AI researcher:

1. Stay up-to-date with the latest research: Keep yourself updated with the latest developments in the field of AI by reading academic papers, attending conferences, and participating in online forums.

2. Develop strong mathematical and programming skills: AI is heavily based on mathematics and programming, so it's essential to have a solid understanding of these subjects. Take courses or self-study to improve your skills in areas such as linear algebra, calculus, probability, and machine learning algorithms.

3. Build a strong network: Building relationships with other researchers, industry professionals, and academics can provide valuable insights and opportunities for collaboration. Attend conferences, join professional organizations

In [ ]:
# Prefer FlashAttention via PyTorch SDPA, else fall back safely.

import torch.nn.functional as F
from torch.nn.attention import sdpa_kernel, SDPBackend


# Example tensors: shape = (batch, heads, seq_len, head_dim)
q = torch.randn(2, 8, 128, 64, device="cuda", dtype=torch.float16)
k = torch.randn(2, 8, 128, 64, device="cuda", dtype=torch.float16)
v = torch.randn(2, 8, 128, 64, device="cuda", dtype=torch.float16)

try:
    with sdpa_kernel(SDPBackend.FLASH_ATTENTION):
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
    chosen_backend = "flash_attention"
except Exception as e:
    with sdpa_kernel([SDPBackend.EFFICIENT_ATTENTION, SDPBackend.MATH]):
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
    chosen_backend = f"fallback ({type(e).__name__})"

print("chosen backend:", chosen_backend)

chosen backend: flash_attention


# SQL generation:

##  Common:

#### SQLFuse:

In [ ]:
def generate_sqlfuse_sqlgen_prompt_dict(
    question: str,
    sqlfuse_question_data: dict,
    schemas_info_all: dict,
    db_id: str = 'mimic_iv',
) -> list[dict[str, str]] | dict[str, str]:
    """
    """

    # Display only the tables that are needed for the question (based on the schema_links)
    schema_info = schemas_info_all[db_id]

    tables = {}
    primary_keys = []
    foreign_keys = []
    one_to_many_relationships = []
    enumeration_values = []  # Need to understand how to get the enumeration values from the schema

    parsed_schema_links = sqlfuse_question_data['sql_fuse_info']['schema_linking_parsed_output']
    table_names = [p_s_l.split(".")[0] for p_s_l in parsed_schema_links["parsed_schema_links"]]
    table_names = list(set(table_names))  # Unique table names

    for table_name in table_names:
        try:
            tables[table_name] = [c['name'] for c in schema_info[table_name]['columns']]
        except KeyError:
            print(sqlfuse_question_data['sql_fuse_info']['schema_linking_output'])
            print(json.dumps(parsed_schema_links, indent=4))
            raise KeyError

        primary_keys.extend([f"{table_name}.{pk}" for pk in schema_info[table_name]['primary_keys']])
        foreign_keys.extend(
            [
                f"{fk['current_table']}.{fk['current_column']}={fk['outer_table']}.{fk['outer_column']}"
                for fk in schema_info[table_name]['foreign_keys']
            ]
        )
        for fk in schema_info[table_name]['foreign_keys']:
            if fk['relationship'] == "1:N":
                one_to_many_relationships.append(
                    f"{fk['current_table']}.{fk['current_column']} has one-to-many relationship with {fk['outer_table']}.{fk['outer_column']}"
                )
            elif fk['relationship'] == "N:1":
                one_to_many_relationships.append(
                    f"{fk['outer_table']}.{fk['outer_column']} has one-to-many relationship with {fk['current_table']}.{fk['current_column']}"
                )

        for c in schema_info[table_name]['columns']:
            if len(c['enum_values']) > 0 and c['display_enum_values_on_prompt']:
                column_enums = []
                for enum in c['enum_values']:
                    if enum['translation']:
                        column_enums.append(f"'{enum['enum_value']}' ({enum.translation})")
                    else:
                        column_enums.append(f"'{enum['enum_value']}'")

                enumeration_values.append(f"{table_name}.{c['name']}={'|'.join(column_enums)}")

    schema_info_str = ""
    for table_name, columns in tables.items():
        schema_info_str += f"{table_name}({'|'.join(columns)})\n"

    return {
        "schema_info": schema_info_str.strip(),
        "primary_keys": "|".join(primary_keys),
        "foreign_keys": "\n".join(foreign_keys),
        "one_to_many_relationships": "\n".join(one_to_many_relationships),
        "enumeration_values": "\n".join(enumeration_values),
        "question": question,
        "schema_linking_columns_description": "\n".join(
            parsed_schema_links["columns"]
            if isinstance(parsed_schema_links["columns"], list)
            else [parsed_schema_links["columns"]]
            ),
        }

#### Prompt creation:

In [ ]:
def create_prompt_system(prompt_sys_template: str, temporal_instructions: str) -> str:
  return prompt_sys_template.format(
      temporal_instructions=temporal_instructions
    )


def create_prompt_user(prompt_user_template: str, sqlfuse_sqlgen_prompt_dict: str, db_engine: str) -> str:
  return prompt_user_template.format(
      db_engine=db_engine,
      schema_info=sqlfuse_sqlgen_prompt_dict['schema_info'],
      primary_keys=sqlfuse_sqlgen_prompt_dict['primary_keys'],
      foreign_keys=sqlfuse_sqlgen_prompt_dict['foreign_keys'],
      one_to_many_relationships=sqlfuse_sqlgen_prompt_dict['one_to_many_relationships'],
      enumeration_values=sqlfuse_sqlgen_prompt_dict['enumeration_values'],
      question=sqlfuse_sqlgen_prompt_dict['question'],
      schema_linking_columns_description=sqlfuse_sqlgen_prompt_dict['schema_linking_columns_description'],
    )

In [ ]:
Role = Literal["system", "user", "assistant"]

def create_prompt_message(role: Role, content: str) -> dict[str,str]:
  return {"role": role, "content": content}


def create_prompt_messages(role_cont_pairs: list[tuple[Role, str]]) -> list[dict[str,str]]:
  return [create_prompt_message(role, content) for role, content in role_cont_pairs]


def create_prompt_from_messages(prompt_messages: list[dict[str,str]], prompt_assistant_think_prefix: str = None) -> str:
  is_think = prompt_assistant_think_prefix is not None
  prompt = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=(not is_think))
  if is_think:
    prompt = prompt + '<|im_start|>assistant\n' + prompt_assistant_think_prefix
  return prompt


def create_prompt_sys_user(prompt_sys: str, prompt_user: str, prompt_assistant_think_prefix: str = None) -> str:
  prompt_messages = create_prompt_messages([("system", prompt_sys), ("user", prompt_user)])
  return create_prompt_from_messages(prompt_messages, prompt_assistant_think_prefix)

#### Retrieving model predictions:

In [ ]:
def extract_sql(model_ans_decoded):
    """
    Extract SQL query from the model output.
    Looks for SQL code within last ```sql ``` code block.
    """
    # Pattern to match SQL code blocks
    pattern = r'```sql\s*(.*?)\s*```'

    # Find all matches (using DOTALL flag to match across newlines)
    matches = re.findall(pattern, model_ans_decoded, re.DOTALL | re.IGNORECASE)

    if matches:
        # Return the last SQL block (typically the final answer)
        sql = matches[-1].strip()
        # Remove comment lines if present
        # sql_lines = [line for line in sql.split('\n') if not line.strip().startswith('--')]
        # return '\n'.join(sql_lines).strip()
        return sql

    return None


def parse_model_ans(model_ans_decoded):
    generated_sql =  extract_sql(model_ans_decoded)
    return model_ans_decoded, generated_sql


def generate_sql(question, sqlfuse_question_data, create_prompt_method, tokenizer, model, max_new_tokens):
  sqlfuse_sql_gen_prompt_dict = generate_sqlfuse_sqlgen_prompt_dict(question, sqlfuse_question_data, sqlfuse_schemas_info)
  prompt_sql_gen = create_prompt_method(sqlfuse_sql_gen_prompt_dict)
  model_ans = model_get_ans(prompt_sql_gen, tokenizer, model, max_new_tokens)
  return parse_model_ans(model_ans)


def generate_fixed_sql(sql_to_fix, error_info, model_full_ans, db_engine, create_prompt_method, tokenizer, model, max_new_tokens):
  prompt_fix_sql = create_prompt_method(sql_to_fix, error_info, model_full_ans, db_engine)
  model_ans = model_get_ans(prompt_fix_sql, tokenizer, model, max_new_tokens)
  return parse_model_ans(model_ans)


def print_parsed_model_ans(ans_full, ans_fetched):
  print(ans_fetched)
  print('-----------------------------\n-----------------------------\n\n')
  print(ans_full)

## SQLite:

#### Prompt templates:

##### Arctic:

In [ ]:
# Arctic, SQLite, no temporal guidance:

ARCTIC_SQLITEGEN_NOTEMP_PROMPT_SYSTEM = """You are a data science expert. Below, you are provided with a database schema and a natural language question. Your task is to understand the schema and generate a valid SQL query to answer the question."""


ARCTIC_SQLITEGEN_NOTEMP_PROMPT_USER = """Database Engine:
{db_engine}

Database Schema:
{schema_info}

Primary keys:
{primary_keys}

Foreign keys:
{foreign_keys}

One-to-many relationships:
{one_to_many_relationships}

Enumeration:
{enumeration_values}

Question:
{question}
{schema_linking_columns_description}

Instructions:
- Make sure you only output the information that is asked in the question. If the question asks for a specific column, make sure to only include that column in the SELECT clause, nothing more.
- The generated query should return all of the information asked in the question without any missing or extra information.
- Before generating the final SQL query, please think through the steps of how to write the query.

Output Format:
Please provide a detailed chain-of-thought reasoning process and include your thought process within ‘<think>‘ tags. Your final answer should be enclosed within ‘<answer>‘ tags.

Ensure that your SQL query follows the correct syntax and is formatted as follows:

```sql
– Your SQL query here
```

Example format:
<think> Step-by-step reasoning, including self-reflection and corrections if necessary. [Limited by 512 tokens] </think>
<answer> Summary of the thought process leading to the final SQL query. [Limited by 512 tokens]

```sql
Correct SQL query here
```
</answer>"""


ARCTIC_SQLITEGEN_NOTEMP_PROMPT_ASSISTANT_PREFIX = "Let me solve this step by step.\n<think>"  #  <-  with thinking


In [ ]:
# Arctic, SQLite, with temporal guidance:

ARCTIC_SQLITEGEN_PROMPT_SYSTEM = """You are a data science expert. Below, you are provided with a database schema and a natural language question. Your task is to understand the schema and generate a valid SQL query to answer the question, according to instructions SQLITE_INSTRUCTIONS:
<SQLITE_INSTRUCTIONS>
{temporal_instructions}
</SQLITE_INSTRUCTIONS>"""


ARCTIC_SQLITEGEN_PROMPT_USER = """Database Engine:
{db_engine}

Database Schema:
{schema_info}

Primary keys:
{primary_keys}

Foreign keys:
{foreign_keys}

One-to-many relationships:
{one_to_many_relationships}

Enumeration:
{enumeration_values}

Question:
{question}
{schema_linking_columns_description}

Instructions:
- Make sure you only output the information that is asked in the question. If the question asks for a specific column, make sure to only include that column in the SELECT clause, nothing more.
- The generated query should return all of the information asked in the question without any missing or extra information.
- The generated query should be written according guidelines from SQLITE_INSTRUCTIONS for time-related predicates in the WHERE clauses.
- Before generating the final SQL query, please think through the steps of how to write the query.

Output Format:
Please provide a detailed chain-of-thought reasoning process and include your thought process within ‘<think>‘ tags. Your final answer should be enclosed within ‘<answer>‘ tags.

Ensure that your SQL query follows the correct syntax and is formatted as follows:

```sql
– Your SQL query here
```

Example format:
<think> Step-by-step reasoning, including self-reflection and corrections if necessary. [Limited by 512 tokens] </think>
<answer> Summary of the thought process leading to the final SQL query. [Limited by 512 tokens]

```sql
Correct SQL query here
```
</answer>"""


ARCTIC_SQLITEGEN_PROMPT_ASSISTANT_PREFIX = """Let me solve this step by step, according to instructions SQLITE_INSTRUCTIONS.
<think>"""


##### Qwen:

In [ ]:
# 3. Qwen, SQLite, no temporal guidance:

QWEN_SQLITEGEN_NOTEMP_PROMPT_SYSTEM = """You are a data science expert. Below, you are provided with a database schema and a natural language question.
Your task is to understand the schema and generate a valid SQL query to answer the question.
Also pay special attention to labels and titles to match exactly to the text of the question."""


QWEN_SQLITEGEN_NOTEMP_PROMPT_USER = """Database Engine:
{db_engine}

Database Schema:
{schema_info}

Primary keys:
{primary_keys}

Foreign keys:
{foreign_keys}

One-to-many relationships:
{one_to_many_relationships}

Enumeration:
{enumeration_values}

Question:
{question}
{schema_linking_columns_description}

Instructions:
- Make sure you only output the information that is asked in the question. If the question asks for a specific column, make sure to only include that column in the SELECT clause, nothing more.
- Make sure that titles and labels are written exactly as they explicitly appear in the question.
- The generated query should return all of the information asked in the question without any missing or extra information.
- The generated query should be a valid SQLite query.
- Before generating the final SQL query, please think through the steps of how to write the query.

Output Format:
Please provide a detailed chain-of-thought reasoning process and include your thought process within ‘<think>‘ tags. Your final answer should be enclosed within ‘<answer>‘ tags.

Ensure that your SQL query follows the correct syntax and is formatted as follows:

```sql
– Your SQL query here
```

Example format:
<think> Step-by-step reasoning, including self-reflection and corrections if necessary. [Limited by 512 tokens] </think>
<answer> Summary of the thought process leading to the final SQL query. [Limited by 512 tokens]

```sql
Correct SQL query here
```
</answer>"""


QWEN_SQLITEGEN_NOTEMP_PROMPT_ASSISTANT_PREFIX = """Let me solve this step by step.
<think>
Taking in account that labels and titles should match the question exactly, the correct SQLite query should be:
```sql"""


In [ ]:
# 4. Qwen, SQlite, with temporal guidance:

QWEN_SQLITEGEN_PROMPT_SYSTEM = """You are a data science expert. Below, you are provided with a database schema and a natural language question.
Your task is to understand the schema and generate a valid SQL query to answer the question, according to instructions SQLITE_INSTRUCTIONS:

<SQLITE_INSTRUCTIONS>
{temporal_instructions}
</SQLITE_INSTRUCTIONS>

Also pay special attention to labels and titles to match exactly to the text of the question."""


QWEN_SQLITEGEN_PROMPT_USER = """Database Engine:
{db_engine}

Database Schema:
{schema_info}

Primary keys:
{primary_keys}

Foreign keys:
{foreign_keys}

One-to-many relationships:
{one_to_many_relationships}

Enumeration:
{enumeration_values}

Question:
{question}
{schema_linking_columns_description}

Instructions:
- Make sure you only output the information that is asked in the question. If the question asks for a specific column, make sure to only include that column in the SELECT clause, nothing more.
- Make sure that titles and labels are written exactly as they explicitly appear in the question.
- The generated query should return all of the information asked in the question without any missing or extra information.
- The generated query should be a valid SQLite query.
- The generated query should be written according guidelines from SQLITE_INSTRUCTIONS for time-related predicates in the WHERE clauses.
- Before generating the final SQL query, please think through the steps of how to write the query.

Output Format:
Please provide a detailed chain-of-thought reasoning process and include your thought process within ‘<think>‘ tags. Your final answer should be enclosed within ‘<answer>‘ tags.

Ensure that your SQL query follows the correct syntax and is formatted as follows:

```sql
– Your SQL query here
```

Example format:
<think> Step-by-step reasoning, including self-reflection and corrections if necessary. [Limited by 512 tokens] </think>
<answer> Summary of the thought process leading to the final SQL query. [Limited by 512 tokens]

```sql
Correct SQL query here
```
</answer>"""


QWEN_SQLITEGEN_PROMPT_ASSISTANT_PREFIX = """Let me solve this step by step.
<think>
According to the provided instructions SQLITE_INSTRUCTIONS, taking in account that labels and titles should match the question exactly, the correct SQLite query should be:
```sql"""


#### Retrieving model predictions:

In [ ]:
# 1. Arctic, SQlite, no temporal guidance:

def create_prompt_sqlitegen_arctic_notemp(sqlfuse_sqlgen_prompt_dict: dict) -> str:
    sql_gen_prompt_user = create_prompt_user(ARCTIC_SQLITEGEN_NOTEMP_PROMPT_USER, sqlfuse_sqlgen_prompt_dict, 'SQLite')
    return create_prompt_sys_user(ARCTIC_SQLITEGEN_NOTEMP_PROMPT_SYSTEM, sql_gen_prompt_user, ARCTIC_SQLITEGEN_NOTEMP_PROMPT_ASSISTANT_PREFIX)

def generate_sqlite_arctic_notemp(question, sqlfuse_question_data, tokenizer, model, max_new_tokens=768+1024):
  return generate_sql(question, sqlfuse_question_data, create_prompt_sqlitegen_arctic_notemp, tokenizer, model, max_new_tokens)

In [ ]:
# 2. Arctic, SQlite, with temporal guidance:

def create_prompt_sqlitegen_arctic(sqlfuse_sqlgen_prompt_dict: dict) -> str:
    sql_gen_prompt_sys = create_prompt_system(ARCTIC_SQLITEGEN_PROMPT_SYSTEM, temp_guide_sqlite)
    sql_gen_prompt_user = create_prompt_user(ARCTIC_SQLITEGEN_PROMPT_USER, sqlfuse_sqlgen_prompt_dict, 'SQLite')
    return create_prompt_sys_user(sql_gen_prompt_sys, sql_gen_prompt_user, ARCTIC_SQLITEGEN_PROMPT_ASSISTANT_PREFIX)

def generate_sqlite_arctic(question, sqlfuse_question_data, tokenizer, model, max_new_tokens=768+1024):
  return generate_sql(question, sqlfuse_question_data, create_prompt_sqlitegen_arctic, tokenizer, model, max_new_tokens)

In [ ]:
# 3.  Qwen, SQlite, no temporal guidance:

def create_prompt_slqitegen_qwen_notemp(sqlfuse_sqlgen_prompt_dict: dict) -> str:
    sql_gen_prompt_user = create_prompt_user(QWEN_SQLITEGEN_NOTEMP_PROMPT_USER, sqlfuse_sqlgen_prompt_dict, 'SQLite')
    return create_prompt_sys_user(QWEN_SQLITEGEN_NOTEMP_PROMPT_SYSTEM, sql_gen_prompt_user, QWEN_SQLITEGEN_NOTEMP_PROMPT_ASSISTANT_PREFIX)

def generate_sqlite_qwen_notemp(question, sqlfuse_question_data, tokenizer, model, max_new_tokens=768+1024):
  return generate_sql(question, sqlfuse_question_data, create_prompt_slqitegen_qwen_notemp, tokenizer, model, max_new_tokens)

In [ ]:
# 4. Qwen, SQlite, with temporal guidance:

def create_prompt_sqlitegen_qwen(sqlfuse_sqlgen_prompt_dict: dict) -> str:
    sql_gen_prompt_sys = create_prompt_system(QWEN_SQLITEGEN_PROMPT_SYSTEM, temp_guide_sqlite)
    sql_gen_prompt_user = create_prompt_user(QWEN_SQLITEGEN_PROMPT_USER, sqlfuse_sqlgen_prompt_dict, 'SQLite')
    return create_prompt_sys_user(sql_gen_prompt_sys, sql_gen_prompt_user, QWEN_SQLITEGEN_PROMPT_ASSISTANT_PREFIX)

def generate_sqlite_qwen(question, sqlfuse_question_data, tokenizer, model, max_new_tokens=768+1024):
  return generate_sql(question, sqlfuse_question_data, create_prompt_sqlitegen_qwen, tokenizer, model, max_new_tokens)

## T-SQL:

#### Prompt templates:

In [ ]:
# 5. Qwen, T-SQL, with temporal guidance (no user-functions):

QWEN_TSQLGEN_NO_USER_FUNC_PROMPT_SYSTEM = """You are an expert in Microsoft SQL Server (MS-SQL Server). Below, you are provided with a database schema and a natural language question.
Your task is to understand the schema and generate a valid MS-SQL Server query (T-SQL or CTE) to answer the question, according to instructions TSQL_INSTRUCTIONS:

<TSQL_INSTRUCTIONS>
{temporal_instructions}
</TSQL_INSTRUCTIONS>

Also pay special attention to labels and titles to match exactly to the text of the question."""


QWEN_TSQLGEN_NO_USER_FUNC_PROMPT_USER = """Database Engine:
{db_engine}

Database Schema:
{schema_info}

Primary keys:
{primary_keys}

Foreign keys:
{foreign_keys}

One-to-many relationships:
{one_to_many_relationships}

Enumeration:
{enumeration_values}

Question:
{question}
{schema_linking_columns_description}

Instructions:
- Make sure you only output the information that is asked in the question. If the question asks for a specific column, make sure to only include that column in the SELECT clause, nothing more.
- Make sure that titles and labels are written exactly as they explicitly appear in the question.
- The generated query should return all of the information asked in the question without any missing or extra information.
- The generated query should be a valid MS-SQL Server query (T-SQL or CTE).
- The generated query should be written according guidelines from TSQL_INSTRUCTIONS for time-related predicates in the WHERE clauses.
- Before generating the final SQL query, please think through the steps of how to write the query.

Output Format:
Please provide a detailed chain-of-thought reasoning process and include your thought process within ‘<think>‘ tags. Your final answer should be enclosed within ‘<answer>‘ tags.

Ensure that your query follows the correct syntax of MS SQL Server (T-SQL or CTE) and is formatted as follows:

```sql
– Your MS SQL Server query here
```

Example format:
<think> Step-by-step reasoning, including self-reflection and corrections if necessary. [Limited by 512 tokens] </think>
<answer> Summary of the thought process leading to the final SQL query. [Limited by 512 tokens]

```sql
Correct MS SQL Server query here
```
</answer>"""


QWEN_TSQLGEN_NO_USER_FUNC_PROMPT_ASSISTANT_PREFIX = """Let me solve this step by step.
<think>
According to the provided instructions TSQL_INSTRUCTIONS, taking in account that labels and titles should match the question exactly, the correct MS-SQL Server query should be:
```sql"""


In [ ]:
# 6. Qwen, T-SQL, with temporal guidance (with user-functions):

QWEN_TSQLGEN_PROMPT_SYSTEM = """You are an expert in Microsoft SQL Server (MS-SQL Server). Below, you are provided with a database schema and a natural language question.
Your task is to understand the schema and generate a valid MS-SQL Server query (T-SQL or CTE) to answer the question, using temporal user-functions according to instructions TSQL_INSTRUCTIONS:

<TSQL_INSTRUCTIONS>
{temporal_instructions}
</TSQL_INSTRUCTIONS>

Also pay special attention to labels and titles to match exactly to the text of the question."""


QWEN_TSQLGEN_PROMPT_USER = """Database Engine:
{db_engine}

Database Schema:
{schema_info}

Primary keys:
{primary_keys}

Foreign keys:
{foreign_keys}

One-to-many relationships:
{one_to_many_relationships}

Enumeration:
{enumeration_values}

Stored user-functions:
See TSQL_INSTRUCTIONS above

Question:
{question}
{schema_linking_columns_description}

Instructions:
- Make sure you only output the information that is asked in the question. If the question asks for a specific column, make sure to only include that column in the SELECT clause, nothing more.
- Make sure that titles and labels are written exactly as they explicitly appear in the question.
- The generated query should return all of the information asked in the question without any missing or extra information.
- The generated query should be a valid MS-SQL Server query (T-SQL or CTE).
- The generated query should use only the `dbo.` user-functions from TSQL_INSTRUCTIONS for time-related predicates in the WHERE clauses.
- Before generating the final SQL query, please think through the steps of how to write the query.

Output Format:
Please provide a detailed chain-of-thought reasoning process and include your thought process within ‘<think>‘ tags. Your final answer should be enclosed within ‘<answer>‘ tags.

Ensure that your query follows the correct syntax of MS SQL Server (T-SQL or CTE) and is formatted as follows:

```sql
– Your MS SQL Server query here
```

Example format:
<think> Step-by-step reasoning, including self-reflection and corrections if necessary. [Limited by 512 tokens] </think>
<answer> Summary of the thought process leading to the final SQL query. [Limited by 512 tokens]

```sql
Correct MS SQL Server query here
```
</answer>"""


QWEN_TSQLGEN_PROMPT_ASSISTANT_PREFIX = """Let me solve this step by step.
<think>
According to the provided instructions TSQL_INSTRUCTIONS, taking in account that labels and titles should match the question exactly, the correct MS-SQL Server query that may use temporal user-functions should be:
```sql"""


#### Retrieving model predictions:

In [ ]:
# 5. Qwen, T-SQL, with temporal guidance (no user-functions):

def create_prompt_tsqlgen_no_user_func_qwen(sqlfuse_sqlgen_prompt_dict: dict) -> str:
    sql_gen_prompt_sys = create_prompt_system(QWEN_TSQLGEN_NO_USER_FUNC_PROMPT_SYSTEM, temp_guide_tsql_no_user_func)
    sql_gen_prompt_user = create_prompt_user(QWEN_TSQLGEN_NO_USER_FUNC_PROMPT_USER, sqlfuse_sqlgen_prompt_dict, 'MS-SQL Server')
    return create_prompt_sys_user(sql_gen_prompt_sys, sql_gen_prompt_user, QWEN_TSQLGEN_NO_USER_FUNC_PROMPT_ASSISTANT_PREFIX)

def generate_tsql_no_user_func_qwen(question, sqlfuse_question_data, tokenizer, model, max_new_tokens=768+1024):
  return generate_sql(question, sqlfuse_question_data, create_prompt_tsqlgen_no_user_func_qwen, tokenizer, model, max_new_tokens)

In [ ]:
# 6. Qwen, T-SQL, with temporal guidance (with user-functions):

def create_prompt_tsqlgen_with_user_func_qwen(sqlfuse_sqlgen_prompt_dict: dict) -> str:
    sql_gen_prompt_sys = create_prompt_system(QWEN_TSQLGEN_PROMPT_SYSTEM, temp_guide_tsql_with_user_func)
    sql_gen_prompt_user = create_prompt_user(QWEN_TSQLGEN_PROMPT_USER, sqlfuse_sqlgen_prompt_dict, 'MS-SQL Server')
    return create_prompt_sys_user(sql_gen_prompt_sys, sql_gen_prompt_user, QWEN_TSQLGEN_PROMPT_ASSISTANT_PREFIX)

def generate_tsql_with_user_func_qwen(question, sqlfuse_question_data, tokenizer, model, max_new_tokens=768+1024):
  return generate_sql(question, sqlfuse_question_data, create_prompt_tsqlgen_with_user_func_qwen, tokenizer, model, max_new_tokens)

## Correcting generated SQL:

In [ ]:
SQL_CORRECT_PROMPT_USER = """Input SQL:
```sql
{sql_to_fix}
```

Error information:
{error_info}.

Please correct the input SQL based on the previous context.
Make sure that you follow all the instructions in the previous context and output a valid {db_engine} query.
Make sure that titles, labels, column names and method names are correct.
Make sure that usage of methods is correct.
Output your reasoning process followed by only one corrected SQL query in the following format:

<think> Step-by-step reasoning and self-reflection regarding the previous context, the input SQL and the error information. [Limited by 512 tokens] </think>
<answer> Summary of the thought process leading to the final corrected SQL query. [Limited by 512 tokens]
```sql
Corrected SQL query here
```
</answer>

Do not output multiple SQLs or only an analysis without a final corrected SQL."""


SQL_CORRECT_PROMPT_ASSISTANT_PREFIX = """Let me solve this step by step.
<think>"""

In [ ]:
# For T-SQL with user-functions

SQL_CORRECT_PROMPT_USER = """Input SQL:
```sql
{sql_to_fix}
```

Error information:
{error_info}.

Please correct the input SQL based on the previous context.
Make sure that you follow all the instructions TSQL_INSTRUCTIONS, if relavant
Make sure that you output a valid {db_engine} query.
Make sure that titles, labels, column names and method names are correct.
Make sure that usage of user and native functions is correct.
Output your reasoning process followed by only one corrected SQL query in the following format:

<think> Step-by-step reasoning and self-reflection regarding the previous context, the input SQL and the error information. [Limited by 512 tokens] </think>
<answer> Summary of the thought process leading to the final corrected SQL query. [Limited by 512 tokens]
```sql
Corrected SQL query here
```
</answer>

Do not output multiple SQLs or only an analysis without a final corrected SQL."""


SQL_CORRECT_PROMPT_ASSISTANT_PREFIX = """Let me solve this step by step, taking in account TSQL_INSTRUCTIONS if needed.
<think>"""

In [ ]:
def parse_full_ans_to_messages(model_full_ans: str) -> list[dict[str, str]]:
  up_to_assistant_text = model_full_ans.split('assistant\n')[0].strip()
  assistant_text = model_full_ans.split('assistant\n')[1].strip()

  up_to_user_text = up_to_assistant_text.split('user\n')[0].strip()
  user_text = up_to_assistant_text.split('user\n')[1].strip()

  system_text = up_to_user_text.split('system\n')[1].strip()

  return create_prompt_messages([('system', system_text), ('user', user_text), ('assistant', assistant_text)])


def create_prompt_fix_sql(sql_to_fix: str, error_info: str, model_full_ans: str, db_engine: str) -> str:
    prompt_messages = parse_full_ans_to_messages(model_full_ans)
    fix_sql_prompt_user = SQL_CORRECT_PROMPT_USER.format(sql_to_fix=sql_to_fix, error_info=error_info, db_engine=db_engine)
    prompt_messages.append(create_prompt_message('user', fix_sql_prompt_user))
    return create_prompt_from_messages(prompt_messages, SQL_CORRECT_PROMPT_ASSISTANT_PREFIX)


def fix_sql(sql_to_fix, error_info, model_full_ans, db_engine, tokenizer, model, max_new_tokens=768+1024):
  return generate_fixed_sql(sql_to_fix, error_info, model_full_ans, db_engine, create_prompt_fix_sql, tokenizer, model, max_new_tokens)

# Playground:

### SQL generation:

In [ ]:
# Below IDs are from ehrsql_2024 VALID:
question_id_1 = "b9c136c1e1d19649caabdeb4"
question_1 = "What is patient 10021487's monthly bilirubin, direct levels since 05/2100? If multiple bilirubin, direct levels are measured monthly, take the average."

question_id_2 = "48f94d76de26cca5b25ee77f"
question_2 = "What are the three most frequently ordered medications for patients diagnosed with acquired absence of organ, genital organs previously within the same hospital visit, during this year?"

question_id_3 = "97a564727fd229d0a1d9c3ba"
question_3 = "Tell me the change in the weight of patient 10027602 second measured on the last hospital visit compared to the value first measured on the last hospital visit?"

question_id_4 = "6360cf590c61b892d228aec3"  # slow: loop of self correcting (was)
question_4 = "What is the cost of an operation referred to as other incision of brain?"

# Below IDs are from ehrsql_2024 TEST:
question_id_5 = "905bf1d8d8b2ee5cc48396ca"
question_5 = "Has the prescription of sodium chloride 0.9%, nicardipine iv, or ondansetron been given to patient 10039997 in 2100?"

question_id_6 = "d738685ff207376b36d479fc"
question_6 = "Is the value of patient 10021118's body temperature from second measurement on the last ICU visit less than its first measurement on the last ICU visit?"


question_id = question_id_6
question = question_6
"""
1. generate_sqlite_arctic_notemp,
3. generate_sqlite_qwen_notemp,
4. generate_sqlite_qwen
5. generate_tsql_no_user_func_qwen
6. generate_tsql_with_user_func_qwen
"""
generate_sql_method = generate_tsql_with_user_func_qwen  #   get_amount_of_tokens(txt, tokenizer)

sqlfuse_question_data = sqlfuse_schema_links[question_id]  # schemas_info_all
full_ans, sql = generate_sql_method(question, sqlfuse_question_data, tokenizer, model, 768)
print_parsed_model_ans(full_ans, sql)


Tokens in prompt: 3571

WITH LastICUStay AS (
    SELECT subject_id, MAX(intime) AS last_intime
    FROM icustays
    WHERE subject_id = 10021118
    GROUP BY subject_id
),
FirstAndSecondMeasurements AS (
    SELECT 
        c.subject_id,
        c.charttime,
        c.valuenum,
        ROW_NUMBER() OVER (ORDER BY c.charttime) AS rn
    FROM 
        chartevents c
    JOIN 
        d_items d ON c.itemid = d.itemid
    JOIN 
        icustays i ON c.stay_id = i.stay_id
    WHERE 
        c.subject_id = 10021118
        AND d.label = 'Body Temperature'
        AND i.intime = (SELECT last_intime FROM LastICUStay)
)
SELECT 
    rn,
    valuenum
FROM 
    FirstAndSecondMeasurements
WHERE 
    rn IN (1, 2);
-----------------------------
-----------------------------


system
You are an expert in Microsoft SQL Server (MS-SQL Server). Below, you are provided with a database schema and a natural language question.
Your task is to understand the schema and generate a valid MS-SQL Server query (T

### Correcting generated SQL:

In [ ]:
err_data_folder = f'{drive_base_path}/colab_input/to_fix_error'
err_data_filenames = [f for f in os.listdir(err_data_folder) if os.path.isfile(os.path.join(err_data_folder, f))]
err_data_filenames.sort()

err_data_filename = err_data_filenames[3]
print(err_data_filename)

if '_sqlite_' in err_data_filename:
  db_engine = 'SQLite'
  datatype = 'sqlite'
elif '_tsql_' in err_data_filename:
  db_engine = 'MS-SQL Server'
  datatype = 'tsql'
else:
  raise ValueError(f'Unknown filename: {err_data_filename}')

pred_sql_attr_name = f'{datatype}_predicted'
pred_sql_ans_attr_name = f'{datatype}_predicted_ans'

errs_data = json_load_from_drive( f'to_fix_error/{err_data_filename}')
err_data = errs_data[0]
print(err_data['id'])
print('-----------------------------\n-----------------------------')

# gather params for fix_sql(sql_to_fix, error_info, model_full_ans, db_engine...)
sql_to_fix = err_data[pred_sql_attr_name]
error_info = err_data[pred_sql_ans_attr_name]
model_full_ans = err_data['prediction_full_ans']


max_new_tok = get_amount_of_tokens(err_data['sqlite'], tokenizer) + 1024
full_ans, sql = fix_sql(sql_to_fix, error_info, model_full_ans, db_engine, tokenizer, model, max_new_tok)
print_parsed_model_ans(full_ans, sql)



NameError: name 'drive_base_path' is not defined

# Batch work:

### SQL generation:

In [ ]:
# 1.
# sql_gen_func = generate_sqlite_arctic_notemp
# sql_pred_attr_name = 'sqlite_predicted'

# 2.
# sql_gen_func = generate_sqlite_arctic
# sql_pred_attr_name = 'tsql_predicted'

# -------------------------------------------------------------

# 3.
# sql_gen_func = generate_sqlite_qwen_notemp
# sql_pred_attr_name = 'sqlite_predicted'

# 4.
# sql_gen_func = generate_sqlite_qwen
# sql_pred_attr_name = 'sqlite_predicted'

# 5.
sql_gen_func = generate_tsql_no_user_func_qwen
sql_pred_attr_name = 'tsql_predicted'

# 6.
# sql_gen_func = generate_tsql_with_user_func_qwen
# sql_pred_attr_name = 'tsql_predicted'

preds_filename_suff = sql_gen_func.__name__.replace('generate_', 'predicted_')
preds_filepath = f'ehrsql2024_{datatype}_{preds_filename_suff}.json' sql_gen_func = generate_tsql_no_user_func_qwen

In [ ]:
dataset = json_load_from_drive(dataset_filepath)
missing_from_sqlfuse_data = []
sql_gold_attr_name = 'sqlite'

for q_data in get_progress_bar(dataset, f'Running [{dataset_filepath}] on [{MODEL_NAME}] with [{sql_gen_func.__name__}]'):
  if q_data[sql_gold_attr_name] != 'null': # Instead of unanswerable questions detection
    if q_data['id'] in sqlfuse_schema_links:
      sqlfuse_question_data = sqlfuse_schema_links[q_data['id']]
      max_new_tok = get_amount_of_tokens(q_data[sql_gold_attr_name], tokenizer) + 1024

      full_ans, sql = sql_gen_func(q_data['question'], sqlfuse_question_data, tokenizer, model, max_new_tok)
      q_data[sql_pred_attr_name] = sql
      q_data['prediction_full_ans'] = full_ans  # .split('assistant')[1].strip()

    else:
      missing_from_sqlfuse_data.append(q_data['id'])

print(f'missing_from_sqlfuse_data:\n{missing_from_sqlfuse_data}')

  0%|          | 0/1063 [00:00<?, ?it/s]

missing_from_sqlfuse_data:
[]


In [ ]:
json_list_write(dataset, preds_filepath)
copy_to_drive(preds_filepath)
files.download(preds_filepath)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Correcting generated SQL:

In [ ]:
err_data_folder_name = 'to_fix_error'
# err_data_folder_name = 'to_fix_empty'

In [ ]:
err_data_folder_path = f'{drive_base_path}/colab_input/{err_data_folder_name}'

if err_data_folder_name.endswith('_error'):
    err_or_empty = 'error'
elif err_data_folder_name.endswith('_empty'):
    err_or_empty = 'empty'
else:
    raise ValueError(f'Unknown errors-data folder: {err_data_folder_name}')

err_data_filenames = [f for f in os.listdir(err_data_folder_path) if os.path.isfile(os.path.join(err_data_folder_path, f))]
# err_data_filenames = [f for f in err_data_filenames if '_tsql_with_user_func_' in f]
err_data_filenames.sort()

In [ ]:
print(f"Fixing SQLs with [{err_or_empty}] answer [{len(err_data_filenames)} files to fix]...\n")

for i, errs_data_filename in enumerate(err_data_filenames):
  fixed_filename = errs_data_filename.replace('.json', '_fixed.json')

  if '_sqlite_' in errs_data_filename:
    db_engine = 'SQLite'
    datatype = 'sqlite'
  elif '_tsql_' in errs_data_filename:
    db_engine = 'MS-SQL Server'
    datatype = 'tsql'
  else:
    raise ValueError(f'Unknown filename: {errs_data_filename}')

  sql_pred_attr_name = f'{datatype}_predicted'
  sql_pred_ans_attr_name = f'{datatype}_predicted_ans'

  errs_data = json_load_from_drive(f'{err_data_folder_name}/{errs_data_filename}')
  for err_data in get_progress_bar(errs_data, f'{i + 1}: Fixing [{errs_data_filename}] on [{MODEL_NAME}]'):
    sql_to_fix = err_data[sql_pred_attr_name]
    error_info = err_data[sql_pred_ans_attr_name] if err_or_empty == 'error' else 'Empty result'
    model_full_ans = err_data['prediction_full_ans']
    max_new_tok = get_amount_of_tokens(err_data['sqlite'], tokenizer) + 1024

    full_ans, sql_fixed = fix_sql(sql_to_fix, error_info, model_full_ans, db_engine, tokenizer, model, max_new_tok)
    err_data[sql_pred_attr_name] = sql_fixed
    err_data['prediction_full_ans'] = full_ans

  json_list_write(errs_data, fixed_filename)
  copy_to_drive(fixed_filename, to_folder_path=f'colab_output/fixed_{err_or_empty}')
  print()
  # files.download(fixed_filename)

print(f"\nFinished Fixing SQLs with [{err_or_empty}] answer")

Fixing SQLs with [error] answer [2 files to fix]...



  0%|          | 0/178 [00:00<?, ?it/s]

Writing to [ehrsql2024_test_predicted_tsql_with_user_func_qwen_ans_err_fixed.json]



  0%|          | 0/184 [00:00<?, ?it/s]

Writing to [ehrsql2024_valid_predicted_tsql_with_user_func_qwen_ans_err_fixed.json]


Finished Fixing SQLs with [error] answer


In [ ]:
print('Done')
runtime.unassign()

Done
